In [11]:
import os

# Create the docs folder
os.makedirs("docs", exist_ok=True)

# All 8 documents
docs = {
    "doc_01.txt": """Delivery Policy

Delivery is available in serviceable pin codes within 10–30 minutes.
Standard delivery is free for orders over INR 149.
Orders below INR 149 have a flat delivery fee of INR 25.
Priority delivery costs an additional INR 15.
Delivery is not available outside serviceable pin codes.
""",

    "doc_02.txt": """Returns & Refunds

Groceries and perishables can be returned within 24 hours if they are damaged, spoiled, or incorrect.
Non-perishable packaged products can be returned within 7 days if unopened and resalable.
Refunds are issued to the original payment method within 3–5 business days or instantly to the wallet.
Opened personal care products are non-returnable except in cases of manufacturing defects.
Return pickup is free.
""",

    "doc_03.txt": """Membership

Basic membership is free.
Zepto Pass costs INR 49 per month and includes free standard delivery and 5% off select categories.
Pass+ costs INR 99 per month and includes free priority delivery, 10% off select categories, and 24-hour early access.
Membership can be cancelled anytime, but there is no refund for the current period.
""",

    "doc_04.txt": """Tracking

Customers can view the live rider map from the packed stage until delivery.
The estimated delivery time is updated as the order progresses.
If there is no movement for more than 20 minutes past the original estimated delivery time, contact support.
""",

    "doc_05.txt": """Cancellation

Orders can be cancelled for free before they reach the Packed stage, typically within the first 2 minutes.
After an order is packed, it cannot be cancelled through the app.
If there is a Zepto-side issue, the order may be automatically cancelled and a full refund will be provided.
""",

    "doc_06.txt": """Damaged/Missing Items

Damaged or missing items should be reported within 24 hours using Report an Issue.
Customers can receive a free replacement or full refund without returning the item.
For orders above INR 1000, a photo is required before processing the request.
""",

    "doc_07.txt": """Gift Cards

Gift cards are available in denominations of INR 100, INR 250, INR 500, and INR 1000.
Gift cards are delivered by email or SMS within minutes.
Gift cards are valid for 1 year and have no maintenance fees.
Gift card balance can be combined with one other payment method, but not with another gift card.
Gift cards cannot be exchanged for cash except where required by law.
""",

    "doc_08.txt": """Support Hours

In-app chat support is available 24/7 with an average response time of under 2 minutes.
Email support is available for non-urgent issues and is answered within 24 business hours.
There is no phone support.
"""
}

# Create all 8 text files
for filename, content in docs.items():
    with open(os.path.join("docs", filename), "w", encoding="utf-8") as f:
        f.write(content)

print("All 8 documents created successfully!")

All 8 documents created successfully!


In [12]:
import os

files = sorted(os.listdir("docs"))

print("Files in docs folder:")
for file in files:
    print(file)

print("\nTotal files:", len(files))

Files in docs folder:
doc_01.txt
doc_02.txt
doc_03.txt
doc_04.txt
doc_05.txt
doc_06.txt
doc_07.txt
doc_08.txt

Total files: 8


In [13]:
import os

documents = []

for filename in sorted(os.listdir("docs")):

    filepath = os.path.join("docs", filename)

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append({
        "id": filename,
        "text": text
    })

print("Number of documents loaded:", len(documents))

Number of documents loaded: 8


In [14]:
for document in documents:
    print(document["id"])

doc_01.txt
doc_02.txt
doc_03.txt
doc_04.txt
doc_05.txt
doc_06.txt
doc_07.txt
doc_08.txt


In [15]:
for document in documents:
    print("=" * 50)
    print("Document:", document["id"])
    print(document["text"])

Document: doc_01.txt
Delivery Policy

Delivery is available in serviceable pin codes within 10–30 minutes.
Standard delivery is free for orders over INR 149.
Orders below INR 149 have a flat delivery fee of INR 25.
Priority delivery costs an additional INR 15.
Delivery is not available outside serviceable pin codes.

Document: doc_02.txt
Returns & Refunds

Groceries and perishables can be returned within 24 hours if they are damaged, spoiled, or incorrect.
Non-perishable packaged products can be returned within 7 days if unopened and resalable.
Refunds are issued to the original payment method within 3–5 business days or instantly to the wallet.
Opened personal care products are non-returnable except in cases of manufacturing defects.
Return pickup is free.

Document: doc_03.txt
Membership

Basic membership is free.
Zepto Pass costs INR 49 per month and includes free standard delivery and 5% off select categories.
Pass+ costs INR 99 per month and includes free priority delivery, 10% of

# Module 3 – Zepto Support Assistant

## Introduction

In this module, I will build a small GenAI-powered support assistant
for Zepto.

The system uses a Retrieval-Augmented Generation (RAG) approach.
It contains eight Zepto policy documents that are converted into
embeddings and stored in ChromaDB.

When a user asks a question, LangGraph routes the question to the
appropriate part of the workflow. Policy-related questions use
retrieval to find relevant information from the documents, while
general questions receive a predefined response.

The final answer is returned using a structured Pydantic schema and
exposed through a FastAPI `/ask` endpoint.

The complete workflow is:

Documents → Chunking → Embeddings → ChromaDB → Query → LangGraph
→ Retrieval → Structured Answer → FastAPI

## Setup

Before building the RAG pipeline, I will install and import the
libraries required for document processing, embeddings, ChromaDB,
LangGraph, Pydantic, and FastAPI.

In [16]:
!pip install -q chromadb sentence-transformers langgraph langchain-core pydantic fastapi uvicorn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [17]:
import os
import chromadb

from sentence_transformers import SentenceTransformer

## Task 1.1 – Load the 8 Zepto Policy Documents

The Zepto support assistant uses eight policy documents stored as
`.txt` files inside the `docs` folder.

I will load each file and store its filename and text in a Python
list. The filename will be used as the document ID.

In [18]:
import os

documents = []

for filename in sorted(os.listdir("docs")):
    filepath = os.path.join("docs", filename)

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append({
        "id": filename,
        "text": text
    })

print("Number of documents loaded:", len(documents))

Number of documents loaded: 8


### Interpretation

All eight Zepto policy documents were successfully loaded from the
`docs` folder. Each document has been stored with a unique document
ID and its corresponding text. These documents will now be prepared
for embedding and retrieval.

## Task 1.2 – Chunk the Documents

The documents need to be divided into chunks before creating
embeddings.

Since the eight policy documents are short, each document can be
treated as one chunk. This keeps the implementation simple while
still allowing ChromaDB to retrieve the relevant policy document.

In [19]:
chunks = []

for document in documents:
    chunks.append({
        "chunk_id": document["id"] + "_chunk_01",
        "document_id": document["id"],
        "text": document["text"]
    })

print("Number of chunks:", len(chunks))

Number of chunks: 8


In [20]:
for chunk in chunks:
    print(chunk["chunk_id"], "->", chunk["document_id"])

doc_01.txt_chunk_01 -> doc_01.txt
doc_02.txt_chunk_01 -> doc_02.txt
doc_03.txt_chunk_01 -> doc_03.txt
doc_04.txt_chunk_01 -> doc_04.txt
doc_05.txt_chunk_01 -> doc_05.txt
doc_06.txt_chunk_01 -> doc_06.txt
doc_07.txt_chunk_01 -> doc_07.txt
doc_08.txt_chunk_01 -> doc_08.txt


### Interpretation

Each policy document has been converted into one searchable chunk.
Therefore, the corpus currently contains eight chunks, with each
chunk retaining its original document ID.

## Task 1.3 – Create Embeddings

Embeddings convert text into numerical vectors that capture the
semantic meaning of the text.

I will use the `all-MiniLM-L6-v2` sentence-transformer model to
generate an embedding for each document chunk.

In [21]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(texts)

print("Number of embeddings:", len(embeddings))
print("Embedding size:", len(embeddings[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of embeddings: 8
Embedding size: 384


### Interpretation

The eight policy chunks were converted into numerical embeddings.
Each chunk has a vector representation that can be compared with
a user's question to find semantically similar policy information.

## Task 1.4 – Store Embeddings in ChromaDB

ChromaDB is used as the vector database for this project.

The embeddings, document text, and document IDs will be stored in
a ChromaDB collection called `zepto_support`.

In [22]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="zepto_support"
)

collection.add(
    ids=[chunk["chunk_id"] for chunk in chunks],
    documents=[chunk["text"] for chunk in chunks],
    embeddings=embeddings.tolist(),
    metadatas=[
        {"document_id": chunk["document_id"]}
        for chunk in chunks
    ]
)

print("Documents stored in ChromaDB:", collection.count())

Documents stored in ChromaDB: 8


In [23]:
query = "What is the delivery fee for orders below INR 149?"

query_embedding = embedding_model.encode([query])[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

for i, document in enumerate(results["documents"][0]):
    print("Result", i + 1)
    print("Document:", results["metadatas"][0][i]["document_id"])
    print(document[:200])
    print()

Result 1
Document: doc_01.txt
Delivery Policy

Delivery is available in serviceable pin codes within 10–30 minutes.
Standard delivery is free for orders over INR 149.
Orders below INR 149 have a flat delivery fee of INR 25.
Priori

Result 2
Document: doc_03.txt
Membership

Basic membership is free.
Zepto Pass costs INR 49 per month and includes free standard delivery and 5% off select categories.
Pass+ costs INR 99 per month and includes free priority delive

Result 3
Document: doc_07.txt
Gift Cards

Gift cards are available in denominations of INR 100, INR 250, INR 500, and INR 1000.
Gift cards are delivered by email or SMS within minutes.
Gift cards are valid for 1 year and have no m



## Task 1 – Interpretation

The eight Zepto policy documents were successfully loaded and
converted into searchable chunks. The `all-MiniLM-L6-v2` model
converted each chunk into an embedding, and the embeddings were
stored in the `zepto_support` ChromaDB collection.

A retrieval test using a delivery-related question returned the
relevant delivery policy document, demonstrating that the vector
store can retrieve policy information based on semantic similarity.

# Task 2 – Structured Prompt Template

The support assistant needs a structured prompt for the optional
real-LLM generation step.

The prompt follows the required role–context–task–format–length
structure. It also contains a negative constraint that prevents the
model from using information outside the retrieved context.

A few-shot example is included to demonstrate the expected answering
style.

In [29]:
SUPPORT_PROMPT = """
ROLE:
You are a Zepto customer support assistant.

CONTEXT:
Use only the Zepto policy information provided below.

TASK:
Answer the user's question using only the provided context.
If the context does not contain the answer, clearly say that the
provided context does not contain enough information.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided
context. Do not invent or assume policy details.

FORMAT:
Return a concise answer suitable for a customer.
Include the relevant source document IDs when requested.

LENGTH:
Keep the answer short and clear, preferably within 2-4 sentences.

FEW-SHOT EXAMPLE:
Context:
Standard delivery is free for orders over INR 149.
Orders below INR 149 have a flat delivery fee of INR 25.

Question:
What is the delivery fee for an order below INR 149?

Answer:
Orders below INR 149 have a flat delivery fee of INR 25.

RETRIEVED CONTEXT:
{context}

USER QUESTION:
{query}
"""

print(SUPPORT_PROMPT)


ROLE:
You are a Zepto customer support assistant.

CONTEXT:
Use only the Zepto policy information provided below.

TASK:
Answer the user's question using only the provided context.
If the context does not contain the answer, clearly say that the
provided context does not contain enough information.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided
context. Do not invent or assume policy details.

FORMAT:
Return a concise answer suitable for a customer.
Include the relevant source document IDs when requested.

LENGTH:
Keep the answer short and clear, preferably within 2-4 sentences.

FEW-SHOT EXAMPLE:
Context:
Standard delivery is free for orders over INR 149.
Orders below INR 149 have a flat delivery fee of INR 25.

Question:
What is the delivery fee for an order below INR 149?

Answer:
Orders below INR 149 have a flat delivery fee of INR 25.

RETRIEVED CONTEXT:
{context}

USER QUESTION:
{query}



## Interpretation

The structured prompt contains all required components: role,
context, task, format, and length. It also explicitly prevents the
assistant from using information outside the supplied context and
contains a few-shot question-and-answer example.

This prompt will be used by the optional real-LLM generation branch.
The required mock mode does not make an LLM call.

# Task 3 – LangGraph Workflow

The LangGraph workflow controls how each user query is processed.

The required baseline uses deterministic mock logic. The
`MOCK_LLM` environment variable controls whether the optional
real-LLM generation path is used.

When `MOCK_LLM` is unset or set to `1`, no LLM call is made.

In [30]:
MOCK_LLM = os.getenv("MOCK_LLM", "1")

print("MOCK_LLM =", MOCK_LLM)

MOCK_LLM = 1


## 3.2 Define the LangGraph State

The graph uses a `TypedDict` state to pass information between
nodes.

The state stores the user's query, classification, retrieved
documents, final answer, sources, and confidence.

In [32]:
from typing import TypedDict

In [33]:
class SupportState(TypedDict, total=False):
    query: str
    intent: str
    retrieved_documents: list
    answer: str
    sources: list
    confidence: float

## 3.3 classify_intent Node

The first node classifies the user query.

In the required mock mode, classification uses the specified
keyword heuristic. If the query contains a policy keyword such as
delivery, return, refund, membership, tracking, cancel, gift card,
or support hours, it is classified as `policy_question`.

Otherwise, it is classified as `general_question`.

No LLM call is made in mock mode.

In [34]:
POLICY_KEYWORDS = [
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "cancel",
    "gift card",
    "support hours"
]


def classify_intent(state: SupportState):
    query = state["query"]
    lower_query = query.lower()

    if MOCK_LLM == "1":
        if any(keyword in lower_query for keyword in POLICY_KEYWORDS):
            intent = "policy_question"
        else:
            intent = "general_question"

    else:
        # Optional real-LLM extension
        # Real LLM classification can be added here.
        # The graded baseline does not require it.
        if any(keyword in lower_query for keyword in POLICY_KEYWORDS):
            intent = "policy_question"
        else:
            intent = "general_question"

    return {
        "intent": intent
    }

In [35]:
print(
    classify_intent(
        {"query": "What is the delivery fee?"}
    )
)

print(
    classify_intent(
        {"query": "Tell me a joke"}
    )
)

{'intent': 'policy_question'}
{'intent': 'general_question'}


## 3.4 retrieve_and_answer Node

For policy questions, the query is converted into an embedding and
sent to ChromaDB.

The top three most similar chunks are retrieved.

This retrieval step runs in both mock and real modes because local
embeddings and ChromaDB do not require an LLM or network connection.

In mock mode, the final answer is created using the required format:

"Based on the retrieved context: {top_chunk_snippet}"

In [36]:
def retrieve_and_answer(state: SupportState):
    query = state["query"]

    # Retrieval always happens
    query_embedding = embedding_model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=3
    )

    retrieved_documents = results["documents"][0]
    retrieved_metadata = results["metadatas"][0]

    source_ids = [
        metadata["document_id"]
        for metadata in retrieved_metadata
    ]

    top_chunk_snippet = retrieved_documents[0][:200]

    if MOCK_LLM == "1":
        answer = f"Based on the retrieved context: {top_chunk_snippet}"

    else:
        # Optional real-LLM generation extension
        context = "\n\n".join(retrieved_documents)

        prompt = SUPPORT_PROMPT.format(
            context=context,
            query=query
        )

        # A real LLM call can be added here.
        answer = f"Based on the retrieved context: {top_chunk_snippet}"

    return {
        "retrieved_documents": retrieved_documents,
        "answer": answer,
        "sources": source_ids,
        "confidence": 1.0
    }

## 3.5 direct_answer Node

General questions do not require retrieval.

In mock mode, the node returns the required fixed response and
does not call an LLM.

In [37]:
def direct_answer(state: SupportState):
    if MOCK_LLM == "1":
        answer = "I can only answer questions about Zepto policies right now."

    else:
        # Optional real-LLM direct-answer extension
        answer = "I can only answer questions about Zepto policies right now."

    return {
        "answer": answer,
        "sources": [],
        "confidence": 1.0
    }

## 3.6 Conditional Routing

After the `classify_intent` node, LangGraph uses a conditional edge.

A `policy_question` is routed to `retrieve_and_answer`.

A `general_question` is routed to `direct_answer`.

The routing itself does not depend on `MOCK_LLM`.

In [38]:
def route_question(state: SupportState):
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"

    return "direct_answer"

In [40]:
from langgraph.graph import StateGraph, START, END

In [41]:
!pip install langgraph

In [42]:
from langgraph.graph import StateGraph, START, END

In [43]:
graph_builder = StateGraph(SupportState)

graph_builder.add_node("classify_intent", classify_intent)
graph_builder.add_node("retrieve_and_answer", retrieve_and_answer)
graph_builder.add_node("direct_answer", direct_answer)

graph_builder.add_edge(START, "classify_intent")

graph_builder.add_conditional_edges(
    "classify_intent",
    route_question,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)

graph_builder.add_edge("retrieve_and_answer", END)
graph_builder.add_edge("direct_answer", END)

graph = graph_builder.compile()

print("LangGraph compiled successfully.")

LangGraph compiled successfully.


## 3.8 Test Both LangGraph Routes

The graph will now be tested with two queries.

The first query is a policy question and should use retrieval.

The second query is unrelated to Zepto policy and should use the
direct-answer route.

In [44]:
policy_result = graph.invoke({
    "query": "What is the delivery fee for orders below INR 149?"
})

general_result = graph.invoke({
    "query": "What is the capital of France?"
})

print("POLICY QUESTION")
print(policy_result)

print("\nGENERAL QUESTION")
print(general_result)

POLICY QUESTION
{'query': 'What is the delivery fee for orders below INR 149?', 'intent': 'policy_question', 'retrieved_documents': ['Delivery Policy\n\nDelivery is available in serviceable pin codes within 10–30 minutes.\nStandard delivery is free for orders over INR 149.\nOrders below INR 149 have a flat delivery fee of INR 25.\nPriority delivery costs an additional INR 15.\nDelivery is not available outside serviceable pin codes.\n', 'Membership\n\nBasic membership is free.\nZepto Pass costs INR 49 per month and includes free standard delivery and 5% off select categories.\nPass+ costs INR 99 per month and includes free priority delivery, 10% off select categories, and 24-hour early access.\nMembership can be cancelled anytime, but there is no refund for the current period.\n', 'Gift Cards\n\nGift cards are available in denominations of INR 100, INR 250, INR 500, and INR 1000.\nGift cards are delivered by email or SMS within minutes.\nGift cards are valid for 1 year and have no main

## Task 3 – Interpretation

The LangGraph workflow successfully routes queries based on their
intent.

The delivery question is classified as `policy_question` and is sent
to `retrieve_and_answer`, where the top three ChromaDB results are
retrieved.

The unrelated question is classified as `general_question` and is
sent directly to `direct_answer` without retrieval.

The default mock mode performs all generation deterministically
without making an LLM call.

# Task 4 – Structured Pydantic Output

The final response must follow a validated JSON structure.

A Pydantic model is used to enforce the required fields:

- `answer`: the final response string
- `sources`: the retrieved chunk or document IDs
- `confidence`: a value between 0 and 1

In [46]:
from pydantic import BaseModel, Field

In [47]:
class SupportResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence: float = Field(ge=0.0, le=1.0)

In [48]:
def create_response(result):
    response = SupportResponse(
        answer=result["answer"],
        sources=result.get("sources", []),
        confidence=result.get("confidence", 1.0)
    )

    return response

In [49]:
response = create_response(policy_result)

print(response.model_dump_json(indent=2))

{
  "answer": "Based on the retrieved context: Delivery Policy\n\nDelivery is available in serviceable pin codes within 10–30 minutes.\nStandard delivery is free for orders over INR 149.\nOrders below INR 149 have a flat delivery fee of INR 25.\nPriori",
  "sources": [
    "doc_01.txt",
    "doc_03.txt",
    "doc_07.txt"
  ],
  "confidence": 1.0
}


In [50]:
{
  "answer": "Based on the retrieved context: ...",
  "sources": [
    "doc_01.txt"
  ],
  "confidence": 1.0
}

{'answer': 'Based on the retrieved context: ...',
 'sources': ['doc_01.txt'],
 'confidence': 1.0}

## Task 4 – Interpretation

The graph output is converted into a Pydantic `SupportResponse`
object.

Pydantic validates that the answer is a string, sources is a list
of strings, and confidence is between 0 and 1.

In mock mode, the response is deterministic and does not require
LLM output validation or retries.

# Task 5 – FastAPI `/ask` Endpoint

The LangGraph support assistant will be exposed through a FastAPI
POST endpoint.

The endpoint accepts a JSON request containing a `query` field and
returns the validated Pydantic response.

In [51]:
from fastapi import FastAPI

app = FastAPI(title="Zepto Support Assistant")


class AskRequest(BaseModel):
    query: str


@app.post("/ask", response_model=SupportResponse)
def ask(request: AskRequest):

    result = graph.invoke({
        "query": request.query
    })

    return create_response(result)

In [52]:
test_request = AskRequest(
    query="What is the delivery fee?"
)

result = ask(test_request)

print(result.model_dump_json(indent=2))

{
  "answer": "Based on the retrieved context: Delivery Policy\n\nDelivery is available in serviceable pin codes within 10–30 minutes.\nStandard delivery is free for orders over INR 149.\nOrders below INR 149 have a flat delivery fee of INR 25.\nPriori",
  "sources": [
    "doc_01.txt",
    "doc_03.txt",
    "doc_07.txt"
  ],
  "confidence": 1.0
}


## Task 5 – Interpretation

The FastAPI application exposes a POST `/ask` endpoint.

The endpoint accepts a Pydantic request containing a user query,
passes the query through the LangGraph workflow, and returns the
validated `SupportResponse`.

Two example calls should be demonstrated for the final submission:
one policy question that triggers retrieval and one general question
that does not trigger retrieval.

uvicorn main:app --host 0.0.0.0 --port 7860

# Task 6 – Docker

A Dockerfile will package the FastAPI application so that it can be
built and run locally.

The container will start the FastAPI application using Uvicorn on
port 7860.

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]

## Task 6 – Interpretation

The Dockerfile defines a reproducible environment for the FastAPI
support assistant.

The application is started using Uvicorn and listens on port 7860.
The required baseline is to build and run the Docker container
locally. Deployment to Hugging Face Spaces is optional.

# Task 7 – README and Architecture

The README documents the complete RAG architecture and explains how
data flows through the support assistant.

The pipeline consists of four main stages:

1. Ingestion
2. Embedding
3. Retrieval
4. Generation

Zepto Policy Documents
        |
        v
Document Ingestion
(docs/*.txt)
        |
        v
Chunking
(per-document chunks)
        |
        v
all-MiniLM-L6-v2
        |
        v
ChromaDB
zepto_support collection
        |
        v
User Query
        |
        v
LangGraph
classify_intent
        |
        +-----------------------+
        |                       |
        v                       v
policy_question          general_question
        |                       |
        v                       v
retrieve_and_answer       direct_answer
        |
        v
Top-3 ChromaDB chunks
        |
        v
Final Answer
        |
        v
Pydantic Response
        |
        v
FastAPI /ask

## Task 7 – Interpretation

The README explains the complete flow from the eight source documents
through ingestion, embedding, ChromaDB retrieval, LangGraph routing,
answer generation, Pydantic validation, and FastAPI.

The README also documents the `MOCK_LLM` behavior. In the default
mock mode, classification and answer generation use deterministic
local logic and do not require an LLM API call. In the optional
real-LLM mode, the generation steps can be connected to an LLM while
retrieval continues to use the local embedding model and ChromaDB.